# 2 - Fine-tune Qwen2-VL-2B (QLoRA) to recognise a crash

The project's VLM scout asks *"is a vehicle crash happening in this image?"*. Out of the box, Qwen2-VL-2B answers
that correctly for only about **39% of real crash frames** (precision 0.88). This notebook teaches it with
**QLoRA** (4-bit base model + small LoRA adapter) on public crash / no-crash frames and **measures the change**:

* the **zero-shot** model (adapter switched off) and the **fine-tuned** model are scored on the same images,
* scores are the model's *Yes minus No* log-odds, so a **threshold** is tuned on validation data only,
* **data protocol:** train on **all of dataset 1**; **dataset 2** (a different source) is cut in half - one half tunes
  the thresholds, the other half gives the final numbers (`test_EXTERNAL`). Near-copies of earlier data are removed
  from each evaluation set. Attach/download **both** datasets: with only one, the notebook falls back to a leaky
  split and says so.

Outputs: the LoRA adapter (`qwen2vl_crash_lora/`), `metrics.json`, and a usage note. The adapter is small (tens of MB).

**Time:** about 1.5-2.5 h with the defaults (3000 training images, 1 epoch). Use `QUICK = True` first (~10 min).
Training is capped by `TIME_LIMIT_MIN` **and** by Kaggle's 12 h limit (an 11 h deadline, keeping back the time the
second scoring pass needs). A resumable checkpoint is written every `CKPT_EVERY_MIN` minutes; if the session dies,
attach the earlier output as data and set `RESUME_FROM`. Training stops with a clear error if the loss diverges.

**Honest limits.** Public crash datasets are mostly CCTV / dashcam stills; results on other camera types may
differ, and datasets labelled by folder name contain label noise. Check the printed dataset summary.

### How to run on Kaggle
1. **Settings (right panel):** Accelerator = **GPU T4 x2** (or P100), **Internet = On**.
2. **Run All** to test, or **Save Version -> Save & Run All (Commit)** so the outputs survive after the session.
3. Everything you need to keep is written to `/kaggle/working/outputs/` (also zipped at the end). Download it from the notebook's **Output** tab.


In [ ]:
!pip install -q -U kagglehub peft bitsandbytes accelerate pillow scikit-learn pandas
!pip install -q -U "transformers>=4.49"
import os, sys, json, time, random, re, glob, math, shutil
from pathlib import Path

ROOT = Path(os.environ.get("WORK", "/kaggle/working"))   # Kaggle's writable folder
OUT = ROOT / "outputs"
OUT.mkdir(parents=True, exist_ok=True)
SEED = 42
random.seed(SEED)

# Kaggle stops a session after 12 h. T0 is the clock; every long step checks hours_left().
T0 = time.time()
HARD_LIMIT_H = 11.0          # leave a 1 h margin below Kaggle's 12 h


def hours_left(reserve_h=0.0):
    """Hours still available before the HARD_LIMIT_H deadline, minus a reserve for later steps."""
    return HARD_LIMIT_H - (time.time() - T0) / 3600 - reserve_h

import torch
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU - enable a GPU accelerator")


In [ ]:
# ---------------- settings ----------------
QUICK = False
MODEL = "Qwen/Qwen2-VL-2B-Instruct"
MAX_PIXELS = 256 * 28 * 28        # image budget per picture (= up to 256 visual tokens). Smaller = faster
N_TRAIN_PER_CLASS = 40 if QUICK else 1500
N_EVAL_PER_CLASS = 20 if QUICK else 1000    # images scored per class on each evaluation set (1000 = use all)
EPOCHS = 1
ACCUM = 8                         # gradient accumulation (batch size is 1 image)
LR = 1e-4
TIME_LIMIT_MIN = 150              # upper bound on training time (also capped by the 12 h deadline)
CKPT_EVERY_MIN = 20               # write a resumable checkpoint this often
RESUME = True                     # continue from outputs/ckpt if it exists
RESUME_FROM = None                # new session: folder of an earlier run's ckpt, e.g.
                                  # "/kaggle/input/<earlier-output>/outputs/ckpt"
QUESTION = ("Is a vehicle crash or collision happening or has one just happened in this image? "
            "Answer yes or no.")   # same wording the project's scout uses
import torch
if torch.cuda.is_available() and "T4" not in torch.cuda.get_device_name(0):
    print("NOTE: choose the GPU T4 accelerator - 4-bit training is not expected to work on a P100.")


In [ ]:
# ---------------- data: download, label from folder names, split ----------------
import numpy as np, pandas as pd
from sklearn.model_selection import GroupShuffleSplit, train_test_split

np.random.seed(SEED)

# Datasets to try, in priority order. The FIRST one that downloads is used for train/val;
# the next one(s) become an EXTERNAL test set. A different source is the honest test:
# frames from the same video never end up on both sides of it.
CANDIDATES = [
    "suryaprabhakaran2005/road-accidents-from-cctv-footages-dataset",
    "ckay16/accident-detection-from-cctv-footage",
    "umitka/real-world-vehicle-crash-dataset-balanced",
]
IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
NEG = re.compile(r"(non[\s_-]*accident|no[\s_-]*accident|not[\s_-]*accident|non[\s_-]*crash|"
                 r"no[\s_-]*crash|normal|safe|negative)", re.I)
POS = re.compile(r"(accident|crash|collision|positive)", re.I)


def label_of(rel_parts):
    """1 = crash, 0 = normal, None = cannot tell. Looks at folder names, deepest first."""
    for part in reversed(rel_parts[:-1]):
        if NEG.search(part):      # checked first: "Non Accident" also contains "accident"
            return 0
        if POS.search(part):
            return 1
    return None


def scan(root, name):
    rows, skipped = [], []
    root = Path(root)
    for p in root.rglob("*"):
        if p.suffix.lower() not in IMG_EXT:
            continue
        rel = p.relative_to(root).parts
        y = label_of(rel)
        if y is None:
            skipped.append(str(p))
            continue
        video = re.sub(r"[_\-\s]*\d+$", "", p.stem)       # drop only the LAST number (the frame no.) -> video id
        rows.append(dict(path=str(p), label=y, source=name,
                         group=f"{name}|{p.parent.relative_to(root)}|{video}"))
    return pd.DataFrame(rows), skipped


def fetch(slug):
    import kagglehub
    return kagglehub.dataset_download(slug)


sources = []                                   # (name, root) pairs
for slug in CANDIDATES:
    try:
        sources.append((slug, fetch(slug)))
    except Exception as e:                    # not found / no internet / needs login
        print(f"skip {slug}: {type(e).__name__}: {str(e)[:90]}")
for d in sorted(glob.glob("/kaggle/input/*")):  # datasets attached through "Add data"
    if re.search("accident|crash", d, re.I) and d not in [r for _, r in sources]:
        sources.append((Path(d).name, d))

frames = []
for name, root in sources:
    df, skipped = scan(root, name)
    npos = int(df.label.sum()) if len(df) else 0
    print(f"{name}: {len(df)} labelled images ({npos} crash / {len(df) - npos} normal), "
          f"{len(skipped)} skipped (label unclear)")
    if skipped:
        print("   e.g. skipped:", skipped[:2])
    if len(df) and df.label.nunique() == 2:
        frames.append(df)
assert frames, ("No usable dataset found. Turn Internet ON in the notebook settings, or use "
                "'Add data' to attach one of CANDIDATES, or edit CANDIDATES.")


def split(df, test_size):
    if df.group.nunique() >= 20:                         # video ids recovered: split by video
        a, b = next(GroupShuffleSplit(1, test_size=test_size, random_state=SEED)
                    .split(df, groups=df.group))
        return df.iloc[a], df.iloc[b], True
    print("WARNING: could not recover video ids from the file names -> random frame split. "
          "Frames of one video can land on both sides, so val numbers are OPTIMISTIC; "
          "trust the EXTERNAL test instead.")
    a, b = train_test_split(df.index, test_size=test_size, stratify=df.label, random_state=SEED)
    return df.loc[a], df.loc[b], False


main = frames[0]
if len(frames) > 1:
    # PROTOCOL: train on ALL of dataset 1. Dataset 2 (a different source) is cut in half:
    #   val  -> tune the decision thresholds,   test_ext -> the final numbers, never used for tuning.
    # (Dataset 1 is made of near-identical video frames, so a split of it cannot be a fair test.)
    train, test_in = main, None
    val, test_ext, grouped = split(pd.concat(frames[1:]).reset_index(drop=True), 0.5)
    print("PROTOCOL: train = dataset 1 | val (thresholds) = half of dataset 2 | test = other half of dataset 2")
else:
    print("WARNING: only ONE dataset is available. Falling back to splitting it; its frames come from the same\n"
          "videos, so val/test numbers will be OPTIMISTIC. Attach a second crash dataset for an honest test.")
    trainval, test_in, grouped = split(main, 0.15)
    train, val, _ = split(trainval, 0.15)
    test_ext = None


def cap(df, n_per_class, seed=SEED):
    """Balanced subsample: at most n_per_class of each label."""
    parts = [g.sample(min(len(g), n_per_class), random_state=seed) for _, g in df.groupby("label")]
    return pd.concat(parts).sample(frac=1, random_state=seed).reset_index(drop=True)


def summary(name, df):
    if df is None:
        print(f"{name:14s} none (only one dataset found - there is NO external test)")
    else:
        print(f"{name:14s} {len(df):6d} images  crash={int(df.label.sum()):6d}  "
              f"normal={int((1 - df.label).sum()):6d}  sources={sorted(df.source.unique())}")


for n_, d_ in (("train", train), ("val", val), ("test (same src)", test_in), ("test EXTERNAL", test_ext)):
    if d_ is not None or n_ == "test EXTERNAL":
        summary(n_, d_)


# ---- near-duplicate guard: no evaluation image may be a copy of a training image ----------
# Two public crash datasets are often cut from the same source videos, which would make an
# "external" test quietly optimistic. dHash = 64-bit fingerprint, robust to resizing / JPEG.
from PIL import Image
from concurrent.futures import ThreadPoolExecutor

POP = np.array([bin(i).count("1") for i in range(256)], dtype=np.uint8)


def dhash(path):
    try:
        im = Image.open(path).convert("L").resize((9, 8), Image.BILINEAR)
    except Exception:
        return None
    a = np.asarray(im, dtype=np.int16)
    bits = (a[:, 1:] > a[:, :-1]).flatten()
    return int.from_bytes(np.packbits(bits).tobytes(), "big")


def hashes_of(df):
    with ThreadPoolExecutor(8) as ex:
        return list(ex.map(dhash, df.path))


def drop_near_duplicates(eval_df, ref_hashes, name, max_dist=6):
    """Remove eval images within `max_dist` bits of any training image (and unreadable files)."""
    ref = np.array([h for h in ref_hashes if h is not None], dtype=np.uint64)
    keep = []
    for h in hashes_of(eval_df):
        if h is None:
            keep.append(False)
            continue
        d = POP[np.bitwise_xor(ref, np.uint64(h)).view(np.uint8)].reshape(-1, 8).sum(axis=1).min()
        keep.append(bool(d > max_dist))
    out = eval_df[np.array(keep)].reset_index(drop=True)
    n_rm = len(eval_df) - len(out)
    print(f"{name}: removed {n_rm} of {len(eval_df)} images that are near-copies of training images")
    if len(eval_df) and n_rm / len(eval_df) > 0.25:
        print(f"   WARNING: {name} overlaps heavily with the training data - the datasets share source "
              f"videos. Prefer a different external dataset; treat this test set's numbers with care.")
    return out


def make_eval_sets(train_hashes, n_per_class):
    """Evaluation sets with every near-copy of earlier data removed, in order: train -> val -> test."""
    sets = {"val": drop_near_duplicates(cap(val, n_per_class), train_hashes, "val")}
    ref = list(train_hashes) + hashes_of(sets["val"])          # the test set must not repeat val either
    if test_ext is not None:
        sets["test_EXTERNAL"] = drop_near_duplicates(cap(test_ext, n_per_class), ref, "test_EXTERNAL")
    if test_in is not None:
        sets["test_same_source"] = drop_near_duplicates(cap(test_in, n_per_class), ref, "test_same_source")
    for k, v in sets.items():
        if len(v) < 40:
            print(f"WARNING: '{k}' has only {len(v)} images after cleaning - its numbers are not reliable.")
    return sets


In [ ]:
# ---------------- load the 4-bit model + LoRA ----------------
from PIL import Image
from transformers import AutoProcessor, Qwen2VLForConditionalGeneration, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

proc = AutoProcessor.from_pretrained(MODEL, min_pixels=64 * 28 * 28, max_pixels=MAX_PIXELS)
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
model = Qwen2VLForConditionalGeneration.from_pretrained(
    MODEL, quantization_config=bnb, device_map={"": 0}, torch_dtype=torch.float16)
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
lcfg = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, task_type="CAUSAL_LM",
    # language-model attention + MLP only (the vision tower stays frozen); the regex covers the
    # module names of both older and newer transformers versions
    target_modules=r".*model\.(language_model\.)?layers\.\d+\.(self_attn\.(q|k|v|o)_proj|mlp\.(gate|up|down)_proj)")
model = get_peft_model(model, lcfg)
model.print_trainable_parameters()

PROMPT = proc.apply_chat_template(
    [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": QUESTION}]}],
    add_generation_prompt=True)
tok = proc.tokenizer
YES = [tok.encode(w, add_special_tokens=False)[0] for w in ("Yes", "yes")]
NO = [tok.encode(w, add_special_tokens=False)[0] for w in ("No", "no")]
print("prompt ends with:", repr(PROMPT[-60:]))


In [ ]:
# ---------------- helpers: one training example, one score ----------------
def load_img(path):
    img = Image.open(path).convert("RGB")
    return img


def make_example(path, label):
    ans = "Yes" if label else "No"
    full = PROMPT + ans + "<|im_end|>"
    enc = proc(text=[full], images=[load_img(path)], return_tensors="pt")
    n_ans = len(tok(ans + "<|im_end|>", add_special_tokens=False).input_ids)
    labels = enc.input_ids.clone()
    labels[:, :-n_ans] = -100                      # learn only the answer tokens
    enc["labels"] = labels
    return enc


@torch.no_grad()
def yes_score(path):
    """log-odds of 'Yes' vs 'No' for the next token (higher = more likely a crash)."""
    enc = proc(text=[PROMPT], images=[load_img(path)], return_tensors="pt").to("cuda")
    with torch.autocast("cuda", dtype=torch.float16):
        logits = model(**enc).logits[0, -1].float()
    lse = lambda ids: torch.logsumexp(logits[ids], dim=0)
    return float(lse(YES) - lse(NO))


def score_df(df, desc):
    t0, out = time.time(), []
    for i, p in enumerate(df.path):
        try:
            out.append(yes_score(p))
        except Exception as e:
            out.append(float("nan"))
        if (i + 1) % 100 == 0:
            print(f"  {desc}: {i + 1}/{len(df)}  ({time.time() - t0:.0f}s)")
    return np.array(out)


# sanity check on one example: the supervised tokens must be exactly "Yes<|im_end|>"
ex = make_example(train.path.iloc[0], 1)
print("supervised tokens:", repr(tok.decode(ex["labels"][0][ex["labels"][0] != -100])))


In [ ]:
# ---------------- evaluation helper (same code as in the other notebooks) ----------------
from sklearn.metrics import roc_auc_score, precision_recall_fscore_support, confusion_matrix


def metrics_at(y, score, thr):
    pred = (np.asarray(score) >= thr).astype(int)
    p, r, f, _ = precision_recall_fscore_support(y, pred, average="binary", zero_division=0)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    return dict(precision=round(float(p), 3), recall=round(float(r), 3), f1=round(float(f), 3),
                tp=int(tp), fp=int(fp), fn=int(fn), tn=int(tn), thr=float(thr))


def pick_thresholds(y, score):
    """Two operating points chosen on VALIDATION data only:
       best-F1, and the lowest threshold that keeps precision >= 0.90 (fewer false alarms)."""
    y = np.asarray(y); score = np.asarray(score)
    grid = np.unique(np.quantile(score, np.linspace(0.01, 0.99, 197)))
    best_f1 = max(grid, key=lambda t: metrics_at(y, score, t)["f1"])
    ok = [t for t in grid if metrics_at(y, score, t)["precision"] >= 0.90]
    if not ok:
        print("NOTE: precision 0.90 is not reachable on the validation data - 'precision90' falls back to "
              "the best-F1 threshold, so it is NOT a high-precision operating point here.")
    return dict(best_f1=float(best_f1), precision90=float(min(ok)) if ok else float(best_f1))


def report(name, y, score, thr):
    out = {"n": int(len(y))}
    try:
        out["auc"] = round(float(roc_auc_score(y, score)), 3)
    except ValueError:
        out["auc"] = None
    for k, t in thr.items():
        out[k] = metrics_at(y, score, t)
    print(name, json.dumps(out))
    return out


In [ ]:
# ---------------- BEFORE: zero-shot scores (adapter switched off) ----------------
train_df = cap(train, N_TRAIN_PER_CLASS)
print("fingerprinting the training images for the duplicate check ...")
train_hashes = hashes_of(train_df)
eval_sets = make_eval_sets(train_hashes, N_EVAL_PER_CLASS)

model.eval()
zero_scores = {}
t_zero = time.time()
with model.disable_adapter():
    for name, df in eval_sets.items():
        zero_scores[name] = score_df(df, f"zero-shot {name}")
t_zero = time.time() - t_zero
print(f"zero-shot scoring done in {t_zero / 60:.1f} min (the fine-tuned pass takes about as long)")


In [ ]:
# ---------------- TRAIN (time budget, checkpoints, resume, divergence guard) ----------------
# the second scoring pass takes about as long as the first one: keep that time free
eval_reserve_h = 1.15 * t_zero / 3600 + 0.25
train_budget_min = max(1.0, min(TIME_LIMIT_MIN, hours_left(eval_reserve_h) * 60))
print(f"training images: {len(train_df)} | budget {train_budget_min:.0f} min (cap {TIME_LIMIT_MIN}; "
      f"session clock {hours_left():.1f} h left; {eval_reserve_h:.1f} h kept for scoring)")

params = [p for p in model.parameters() if p.requires_grad]
opt = torch.optim.AdamW(params, lr=LR, weight_decay=0.0)
total_steps = max(1, EPOCHS * len(train_df) // ACCUM)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, total_steps=total_steps + 1,
                                            pct_start=0.1, anneal_strategy="cos")
scaler = torch.amp.GradScaler("cuda")

CKPT = OUT / "ckpt"
if RESUME_FROM and not (CKPT / "state.pt").exists():
    shutil.copytree(RESUME_FROM, CKPT, dirs_exist_ok=True)     # bring an earlier session's checkpoint here
start_ep, start_i, n_seen = 0, 0, 0


def save_ckpt(ep, i):
    model.save_pretrained(str(CKPT / "adapter"))
    torch.save(dict(opt=opt.state_dict(), sched=sched.state_dict(), scaler=scaler.state_dict(),
                    ep=ep, i=i, n_seen=n_seen), CKPT / "state.pt")


if RESUME and (CKPT / "state.pt").exists():
    from peft import set_peft_model_state_dict
    from safetensors.torch import load_file
    set_peft_model_state_dict(model, load_file(str(CKPT / "adapter" / "adapter_model.safetensors")))
    st = torch.load(CKPT / "state.pt", map_location="cpu")
    opt.load_state_dict(st["opt"]); sched.load_state_dict(st["sched"]); scaler.load_state_dict(st["scaler"])
    start_ep, start_i, n_seen = st["ep"], st["i"], st["n_seen"]
    print(f"RESUMED from checkpoint: epoch {start_ep}, image index {start_i}, {n_seen} images already trained")

model.train()
t0, last_ckpt, run_loss, bad, stop = time.time(), time.time(), [], 0, False
for ep in range(start_ep, EPOCHS):
    order = train_df.sample(frac=1, random_state=SEED + ep).reset_index(drop=True)
    for i, (path, label) in enumerate(zip(order.path, order.label)):
        if ep == start_ep and i < start_i:
            continue                                           # done before the checkpoint
        try:
            enc = make_example(path, int(label)).to("cuda")
        except Exception:
            continue                                           # unreadable image
        with torch.autocast("cuda", dtype=torch.float16):
            loss = model(**enc).loss / ACCUM
        if not torch.isfinite(loss):                           # fp16 can blow up: skip; stop if it persists
            bad += 1
            opt.zero_grad(set_to_none=True)
            print(f"non-finite loss at image {n_seen} ({bad} so far) - skipped")
            if bad >= 20:
                save_ckpt(ep, i)
                raise RuntimeError("Training diverged (20 non-finite losses). Lower LR (try 5e-5) or "
                                   "MAX_PIXELS and run again; the last good state is in outputs/ckpt.")
            continue
        scaler.scale(loss).backward()
        run_loss.append(loss.item() * ACCUM)
        n_seen += 1
        if n_seen % ACCUM == 0:
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(params, 1.0)
            scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True)
            if sched.last_epoch < total_steps:
                sched.step()
        if n_seen % 100 == 0:
            print(f"ep {ep} img {n_seen}/{EPOCHS * len(train_df)}  loss(last 100) "
                  f"{np.mean(run_loss[-100:]):.3f}  {(time.time() - t0) / 60:.1f} min")
        if (time.time() - last_ckpt) / 60 > CKPT_EVERY_MIN:
            save_ckpt(ep, i + 1)
            last_ckpt = time.time()
            print(f"checkpoint saved ({(time.time() - t0) / 60:.0f} min)")
        if (time.time() - t0) / 60 > train_budget_min:
            print("time budget reached - stopping training here")
            stop = True
            break
    if stop:
        break
save_ckpt(ep if stop else EPOCHS, i + 1 if stop else 0)
model.eval()
print(f"trained on {n_seen} images in {(time.time() - t0) / 60:.1f} min")


In [ ]:
# ---------------- AFTER: score again, tune thresholds on VAL only, report ----------------
tuned_scores = {name: score_df(df, f"fine-tuned {name}") for name, df in eval_sets.items()}

summary = {}
for tag, scores in (("zero_shot", zero_scores), ("fine_tuned", tuned_scores)):
    ok = ~np.isnan(scores["val"])
    thr = pick_thresholds(eval_sets["val"].label.values[ok], scores["val"][ok])
    summary[tag] = {"thresholds_chosen_on_val": thr}
    for name, df in eval_sets.items():
        ok = ~np.isnan(scores[name])
        summary[tag][name] = report(f"[{tag}] {name}", df.label.values[ok], scores[name][ok], thr)


In [ ]:
# ---------------- save the adapter + metrics + usage note ----------------
ADAPTER = OUT / "qwen2vl_crash_lora"
model.save_pretrained(str(ADAPTER))
proc.save_pretrained(str(ADAPTER))
meta = dict(base_model=MODEL, question=QUESTION, max_pixels=MAX_PIXELS,
            datasets=[n for n, _ in sources], trained_on_images=int(n_seen),
            grouped_split=bool(grouped),
            note="Scores are Yes-vs-No log-odds. Headline = test_EXTERNAL (different dataset), if present.")
(OUT / "metrics.json").write_text(json.dumps(dict(meta=meta, results=summary), indent=2))
(OUT / "HOW_TO_USE_VLM.txt").write_text(f"""Load the 4-bit base model and attach the adapter:

    from transformers import Qwen2VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
    from peft import PeftModel
    import torch
    base = Qwen2VLForConditionalGeneration.from_pretrained("{MODEL}", device_map={{"": 0}},
              quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                                    bnb_4bit_compute_dtype=torch.float16))
    model = PeftModel.from_pretrained(base, "qwen2vl_crash_lora")
    # ask the same question as training: {QUESTION!r}  -> answers 'Yes' / 'No'

With the traffic_watch project:  python -m traffic_watch ... --vlm --vlm-scout --vlm-adapter path/to/qwen2vl_crash_lora
Check metrics.json: judge the model on 'test_EXTERNAL', not on 'val'.
""")
print(json.dumps(summary, indent=1)[:3000])
shutil.make_archive(str(ROOT / "vlm_outputs"), "zip", OUT)
print("Download:", ROOT / "vlm_outputs.zip")
